# Lesson 4.2 — Position Interpolation and the IK-per-Sample Loop

*Module 7 · Unit 4*

**This notebook:** realizes a straight-line Cartesian move via interpolate-then-IK, verifies FK-match + branch consistency, and shows a branch flip causes a joint jump and an out-of-workspace line fails.

Run top to bottom (Restart & Run All). It ends by printing `All checks passed.`

In [ ]:
# --- Module 7 engine (embedded; builds on the M6 velocity layer) ---
"""
Module 7 engine - Trajectory Generation (Installment A: Units 1-2).
Builds ON the Module 6 velocity layer (imported verbatim, not reimplemented).
Greenhouse arm: planar 2R, L1=0.4, L2=0.3 (as in M5/M6); redundant 3R extension.
"""
import numpy as np

# ----- Module 6 base (reused verbatim from lesson32_capstone_velocity_layer) -----
def dh(th, d, a, al):
    ct, st, ca, sa = np.cos(th), np.sin(th), np.cos(al), np.sin(al)
    return np.array([[ct, -st*ca, st*sa, a*ct],
                     [st,  ct*ca, -ct*sa, a*st],
                     [0,      sa,     ca,   d ],
                     [0,       0,      0,   1 ]])

def forward_chain(P, T, q):
    M = np.eye(4); Ms = [M.copy()]
    for i, (th0, d0, a, al) in enumerate(P):
        th, d = (th0+q[i], d0) if T[i] == "R" else (th0, d0+q[i])
        M = M @ dh(th, d, a, al); Ms.append(M.copy())
    return Ms

def geometric_jacobian(P, T, q):
    Ms = forward_chain(P, T, q); on = Ms[-1][:3, 3]; J = np.zeros((6, len(q)))
    for i in range(len(q)):
        z = Ms[i][:3, 2]; o = Ms[i][:3, 3]
        if T[i] == "R": J[:3, i] = np.cross(z, on-o); J[3:, i] = z
        else: J[:3, i] = z
    return J

def Jv_planar(P, T, q): return geometric_jacobian(P, T, q)[:2, :]
def fk_xy(P, T, q): return forward_chain(P, T, q)[-1][:2, 3]

EPS = 0.08  # singularity threshold on sigma_min (M6 convention)

def analyze(P, T, q):
    J = Jv_planar(P, T, q); U, S, Vt = np.linalg.svd(J)
    w = float(np.prod(S)); kappa = float(S[0]/max(S[-1], 1e-12))
    return {"sigma": S, "w": w, "kappa": kappa,
            "axes": [(U[:, i], float(S[i])) for i in range(len(S))],
            "singular": bool(S[-1] < EPS), "sigma_min": float(S[-1]),
            "J": J, "U": U, "Vt": Vt}

def dls(J, lam): return J.T @ np.linalg.inv(J @ J.T + lam**2*np.eye(J.shape[0]))

def schedule_lambda(sigma_min, lam_max=0.1, eps=EPS):
    if sigma_min >= eps: return 0.0
    return float(np.sqrt((1-(sigma_min/eps)**2))*lam_max)

def velocity_layer(P, T, q, xi_d, z=None):
    """M6 deliverable: commanded tool twist -> joint rates (open-loop, singularity-robust)."""
    rep = analyze(P, T, q); lam = schedule_lambda(rep["sigma_min"])
    J = rep["J"]; Jd = dls(J, lam) if lam > 0 else np.linalg.pinv(J)
    qd = Jd @ xi_d
    if z is not None:
        qd = qd + (np.eye(len(q)) - Jd @ J) @ z
    info = {"w": rep["w"], "kappa": rep["kappa"], "sigma_min": rep["sigma_min"],
            "lambda": lam, "singular": rep["singular"]}
    return qd, info

# Canonical greenhouse arms
P2 = [(0, 0, 0.4, 0), (0, 0, 0.3, 0)]; T2 = ["R", "R"]                       # 2R, L1=0.4 L2=0.3
P3 = [(0, 0, 0.4, 0), (0, 0, 0.3, 0), (0, 0, 0.2, 0)]; T3 = ["R", "R", "R"]  # redundant 3R

# ----- Module 7 NEW: time parameterization (Units 1-2) -----
def poly_eval(c, t):
    """Evaluate ascending-coeff polynomial c=[c0..cn] and its 1st-3rd derivatives at t."""
    t = np.asarray(t, float); n = len(c)
    pos = sum(c[k]*t**k for k in range(n))
    vel = sum(k*c[k]*t**(k-1) for k in range(1, n)) if n > 1 else np.zeros_like(t)
    acc = sum(k*(k-1)*c[k]*t**(k-2) for k in range(2, n)) if n > 2 else np.zeros_like(t)
    jrk = sum(k*(k-1)*(k-2)*c[k]*t**(k-3) for k in range(3, n)) if n > 3 else np.zeros_like(t)
    return pos, vel, acc, jrk

def cubic_coeffs(q0, qf, v0, vf, T):
    """Cubic q(t) on [0,T] matching position+velocity endpoints. Ascending coeffs (C1)."""
    a0 = q0; a1 = v0
    a2 = (3*(qf-q0) - (2*v0+vf)*T) / T**2
    a3 = (2*(q0-qf) + (v0+vf)*T) / T**3
    return np.array([a0, a1, a2, a3])

def quintic_coeffs(q0, qf, v0, vf, a0, af, T):
    """Quintic q(t) on [0,T] matching position+velocity+acceleration endpoints. Ascending (C2)."""
    c0 = q0; c1 = v0; c2 = a0/2.0
    T2, T3, T4, T5 = T**2, T**3, T**4, T**5
    h = qf - q0
    c3 = (20*h - (8*vf + 12*v0)*T - (3*a0 - af)*T2) / (2*T3)
    c4 = (-30*h + (14*vf + 16*v0)*T + (3*a0 - 2*af)*T2) / (2*T4)
    c5 = (12*h - (6*vf + 6*v0)*T - (a0 - af)*T2) / (2*T5)
    return np.array([c0, c1, c2, c3, c4, c5])

def trapezoidal_profile(dist, v_max, a_max, n=400):
    """Trapezoidal velocity profile over scalar distance. Returns t, s, v, a, T_total.
    Acceleration is piecewise-constant -> discontinuous (jerk impulses at the corners)."""
    dist = float(dist); sgn = 1.0 if dist >= 0 else -1.0; D = abs(dist)
    if D == 0: t = np.linspace(0,1,n); z=np.zeros_like(t); return t,z,z,z,1.0
    t_acc = v_max / a_max
    d_acc = 0.5 * a_max * t_acc**2
    if 2*d_acc > D:                      # triangular (never reaches v_max)
        t_acc = np.sqrt(D / a_max); v_peak = a_max*t_acc; t_flat = 0.0
    else:
        v_peak = v_max; t_flat = (D - 2*d_acc) / v_max; d_acc = 0.5*a_max*t_acc**2
    Tf = 2*t_acc + t_flat
    t = np.linspace(0, Tf, n)
    s = np.zeros_like(t); v = np.zeros_like(t); a = np.zeros_like(t)
    for i, ti in enumerate(t):
        if ti < t_acc:
            a[i] = a_max; v[i] = a_max*ti; s[i] = 0.5*a_max*ti**2
        elif ti < t_acc + t_flat:
            a[i] = 0.0; v[i] = v_peak; s[i] = d_acc + v_peak*(ti - t_acc)
        else:
            td = ti - t_acc - t_flat
            a[i] = -a_max; v[i] = v_peak - a_max*td
            s[i] = d_acc + v_peak*t_flat + v_peak*td - 0.5*a_max*td**2
    return t, sgn*s, sgn*v, sgn*a, Tf

def s_curve_profile(dist, v_max, a_max, j_max, n=600):
    """Jerk-limited (S-curve) profile by integrating a 7-segment jerk sequence.
    Returns t, s, v, a, j, T_total. Jerk is bounded by j_max -> acceleration is continuous."""
    dist = float(dist); sgn = 1.0 if dist >= 0 else -1.0; D = abs(dist)
    if D == 0: t = np.linspace(0,1,n); z=np.zeros_like(t); return t,z,z,z,z,1.0
    Tj = a_max / j_max
    t_const_a = max(v_max/a_max - Tj, 0.0)
    dt = 5e-4
    def simulate(coast):
        seg = [(+j_max, Tj), (0.0, t_const_a), (-j_max, Tj),
               (0.0, coast),
               (-j_max, Tj), (0.0, t_const_a), (+j_max, Tj)]
        ts=[0.0]; s=[0.0]; v=[0.0]; a=[0.0]; j=[0.0]
        for jv, dur in seg:
            steps = max(int(round(dur/dt)), 0)
            for _ in range(steps):
                a_n = a[-1]+jv*dt; v_n = v[-1]+a[-1]*dt; s_n = s[-1]+v[-1]*dt
                ts.append(ts[-1]+dt); j.append(jv); a.append(a_n); v.append(v_n); s.append(s_n)
        return (np.array(ts),np.array(s),np.array(v),np.array(a),np.array(j))
    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid=(lo+hi)/2; _,s,_,_,_ = simulate(mid)
        if s[-1] < D: lo=mid
        else: hi=mid
    ts,s,v,a,j = simulate((lo+hi)/2)
    tt=np.linspace(0,ts[-1],n)
    s=np.interp(tt,ts,s); v=np.interp(tt,ts,v); a=np.interp(tt,ts,a); j=np.interp(tt,ts,j)
    return tt, sgn*s, sgn*v, sgn*a, sgn*j, float(ts[-1])

# ============================================================================
# Unit 3-4 additions (Installment B): joint-space & Cartesian-space trajectories
# Builds on the M6 base above (fk_xy, velocity_layer, P2/T2) and the Unit 1-2
# time utilities (cubic_coeffs, quintic_coeffs, poly_eval). No dynamics, no feedback.
# ============================================================================

# ---- planar 2R closed-form inverse kinematics (position only) --------------
def ik_2link(x, y, L1=0.4, L2=0.3, elbow="up"):
    """Closed-form IK for the planar 2R arm. Returns (q1, q2) or None if unreachable.
    elbow='up' picks q2>=0; 'down' picks q2<=0."""
    r2 = x*x + y*y
    c2 = (r2 - L1*L1 - L2*L2) / (2*L1*L2)
    if c2 < -1.0 - 1e-9 or c2 > 1.0 + 1e-9:
        return None                      # outside the annulus -> unreachable
    c2 = min(1.0, max(-1.0, c2))
    s2 = np.sqrt(max(0.0, 1 - c2*c2))
    if elbow == "down": s2 = -s2
    q2 = np.arctan2(s2, c2)
    q1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(q2), L1 + L2*np.cos(q2))
    return np.array([q1, q2])

# ---- synchronized multi-joint polynomial trajectory ------------------------
def joint_traj(q0, qf, T, kind="quintic", v0=None, vf=None):
    """Per-joint polynomial coefficients for a synchronized rest-to-rest move
    over a COMMON duration T (all joints start and finish together).
    kind in {'cubic','quintic'}. Returns a list of coefficient arrays."""
    q0 = np.atleast_1d(np.asarray(q0, float)); qf = np.atleast_1d(np.asarray(qf, float))
    n = len(q0); v0 = np.zeros(n) if v0 is None else np.atleast_1d(v0)
    vf = np.zeros(n) if vf is None else np.atleast_1d(vf)
    coeffs = []
    for i in range(n):
        if kind == "cubic":
            coeffs.append(cubic_coeffs(q0[i], qf[i], v0[i], vf[i], T))
        else:
            coeffs.append(quintic_coeffs(q0[i], qf[i], v0[i], vf[i], 0.0, 0.0, T))
    return coeffs

def sample_joint_traj(coeffs, T, n=200):
    """Sample a multi-joint polynomial trajectory. Returns t, Q, Qd, Qdd (each n x dof)."""
    t = np.linspace(0, T, n)
    cols = [poly_eval(c, t) for c in coeffs]
    Q   = np.stack([c[0] for c in cols], axis=1)
    Qd  = np.stack([c[1] for c in cols], axis=1)
    Qdd = np.stack([c[2] for c in cols], axis=1)
    return t, Q, Qd, Qdd

def sync_duration(q0, qf, vmax):
    """Minimum common duration so no joint exceeds vmax under a quintic
    (peak speed = 15*|dq|/(8T)). The slowest joint sets the pace."""
    q0 = np.atleast_1d(np.asarray(q0, float)); qf = np.atleast_1d(np.asarray(qf, float))
    dq = np.abs(qf - q0)
    return float(np.max(15.0 * dq / (8.0 * vmax)))

# ---- C2 natural cubic spline through via-points (per scalar joint) ---------
def cubic_spline_natural(ts, ys):
    """Natural cubic spline (second derivative = 0 at the ends) through knots
    (ts, ys). Returns a dict usable by eval_spline. C2 at interior knots."""
    ts = np.asarray(ts, float); ys = np.asarray(ys, float); n = len(ts) - 1
    h = np.diff(ts)
    # solve for second derivatives m via tridiagonal system (natural BCs m0=mn=0)
    A = np.zeros((n+1, n+1)); rhs = np.zeros(n+1)
    A[0,0] = 1.0; A[n,n] = 1.0
    for i in range(1, n):
        A[i, i-1] = h[i-1]
        A[i, i]   = 2*(h[i-1] + h[i])
        A[i, i+1] = h[i]
        rhs[i] = 6*((ys[i+1]-ys[i])/h[i] - (ys[i]-ys[i-1])/h[i-1])
    m = np.linalg.solve(A, rhs)
    return {"ts": ts, "ys": ys, "m": m, "h": h}

def eval_spline(sp, t):
    """Evaluate a natural cubic spline and its first two derivatives at scalar t.
    Returns (y, yd, ydd)."""
    ts, ys, m, h = sp["ts"], sp["ys"], sp["m"], sp["h"]
    i = int(np.clip(np.searchsorted(ts, t) - 1, 0, len(ts)-2))
    dt = t - ts[i]; hi = h[i]
    A = (ts[i+1]-t)/hi; B = (t-ts[i])/hi
    y = (A*ys[i] + B*ys[i+1]
         + ((A**3 - A)*m[i] + (B**3 - B)*m[i+1]) * (hi*hi)/6.0)
    yd = ((ys[i+1]-ys[i])/hi
          - (3*A*A - 1)/6.0*hi*m[i] + (3*B*B - 1)/6.0*hi*m[i+1])
    ydd = A*m[i] + B*m[i+1]
    return y, yd, ydd

# ---- Cartesian straight-line tool motion + IK-per-sample -------------------
def cartesian_line(p0, p1, s):
    """Straight-line tool position at path parameter s in [0,1]."""
    p0 = np.asarray(p0, float); p1 = np.asarray(p1, float)
    return (1.0 - s)*p0 + s*p1

def cartesian_traj_ik(p0, p1, T, n=120, kind="quintic", elbow="up", L1=0.4, L2=0.3):
    """Straight-line tool path p0->p1, timed by a quintic time scaling s(t),
    with closed-form 2R IK solved at each sample. Returns t, Ptool (n x 2),
    Q (n x 2). Raises ValueError if any sample is unreachable."""
    t = np.linspace(0, T, n)
    cs = quintic_coeffs(0.0, 1.0, 0, 0, 0, 0, T)
    s, _, _, _ = poly_eval(cs, t)
    Ptool = np.stack([cartesian_line(p0, p1, si) for si in s])
    Q = np.zeros((n, 2))
    for k in range(n):
        sol = ik_2link(Ptool[k,0], Ptool[k,1], L1, L2, elbow)
        if sol is None:
            raise ValueError("unreachable tool point on the straight-line path at sample %d" % k)
        Q[k] = sol
    return t, Ptool, Q

# ---- orientation interpolation: SLERP --------------------------------------
def quat_axis_angle(axis, angle):
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.concatenate([[np.cos(angle/2)], np.sin(angle/2)*axis])  # [w, x, y, z]

def slerp(q0, q1, s):
    """Spherical linear interpolation between unit quaternions q0, q1 (w,x,y,z)."""
    q0 = np.asarray(q0, float); q1 = np.asarray(q1, float)
    q0 = q0/np.linalg.norm(q0); q1 = q1/np.linalg.norm(q1)
    d = np.dot(q0, q1)
    if d < 0:                       # take the shortest arc
        q1 = -q1; d = -d
    if d > 0.9995:                  # nearly parallel -> linear, then renormalize
        q = q0 + s*(q1 - q0); return q/np.linalg.norm(q)
    th0 = np.arccos(d); th = th0*s
    q2 = q1 - q0*d; q2 = q2/np.linalg.norm(q2)
    return q0*np.cos(th) + q2*np.sin(th)

def slerp_angle(a0, a1, s):
    """Shortest-arc interpolation of a planar orientation angle (radians)."""
    d = (a1 - a0 + np.pi) % (2*np.pi) - np.pi   # wrap to (-pi, pi]
    return a0 + s*d

# ---- screw interpolation in SE(2) (unified position + orientation) ---------
def se2(theta, x, y):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, x], [s, c, y], [0, 0, 1.0]])

def _se2_log(Tm):
    """Matrix log of an SE(2) element -> twist (vx, vy, omega)."""
    th = np.arctan2(Tm[1,0], Tm[0,0]); p = Tm[:2, 2]
    if abs(th) < 1e-9:
        return np.array([p[0], p[1], 0.0])
    A = np.sin(th)/th; B = (1 - np.cos(th))/th
    Vinv = np.array([[A, B], [-B, A]]) / (A*A + B*B)
    v = Vinv @ p
    return np.array([v[0], v[1], th])

def _se2_exp(xi):
    """Exp of an se(2) twist (vx, vy, omega) -> SE(2) element."""
    vx, vy, th = xi
    if abs(th) < 1e-9:
        return se2(0.0, vx, vy)
    V = np.array([[np.sin(th), -(1-np.cos(th))], [(1-np.cos(th)), np.sin(th)]]) / th
    p = V @ np.array([vx, vy])
    return se2(th, p[0], p[1])

def screw_interp_se2(T0, T1, s):
    """Constant-twist (screw) interpolation between SE(2) poses T0, T1 at s in [0,1].
    T(s) = T0 * exp(s * log(T0^{-1} T1)) -> a single screw motion."""
    T0inv = np.linalg.inv(T0)
    xi = _se2_log(T0inv @ T1)
    return T0 @ _se2_exp(s * xi)


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
checks = []

In [ ]:
p0=np.array([0.5,0.1]); p1=np.array([0.2,0.35])
t,P,Q=cartesian_traj_ik(p0,p1,1.0,n=80,elbow='up')
# every sample FK-matches its tool point; path straight
fk_ok=all(np.linalg.norm(fk_xy(P2,T2,Q[k])-P[k])<1e-9 for k in range(len(t)))
checks.append(fk_ok)
d=p1-p0; d=d/np.linalg.norm(d); dev=max(np.linalg.norm((P[k]-p0)-np.dot(P[k]-p0,d)*d) for k in range(len(P)))
checks.append(dev<1e-12)
# branch-consistent joint path: no large jumps between consecutive samples
jump=np.max([np.linalg.norm(Q[k+1]-Q[k]) for k in range(len(Q)-1)])
checks.append(jump<0.2)
print('max consecutive joint step (consistent branch): %.4f rad'%jump)

In [ ]:
# deliberately flip the branch at the midpoint -> joint jump appears
Qflip=Q.copy(); mid=len(Q)//2
for k in range(mid,len(Q)):
    Qflip[k]=ik_2link(*P[k],elbow='down')
jump_flip=np.max([np.linalg.norm(Qflip[k+1]-Qflip[k]) for k in range(len(Qflip)-1)])
print('max joint step WITH a branch flip: %.4f rad (a jump)'%jump_flip)
checks.append(jump_flip>0.5)          # flip causes a large jump
# an out-of-workspace straight line must fail cleanly
raised=False
try:
    cartesian_traj_ik([0.69,0.0],[-0.69,0.0],1.0,n=40,elbow='up')
except ValueError:
    raised=True
checks.append(raised)
print('out-of-workspace line raised ValueError:',raised)

In [ ]:
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')